# Session 1: a shape is a function
**A. Real-world problem.** CAD software has to answer one question millions of times: "is
this point inside the part?" A mug is not stored as a picture. It can be stored as a
*function*: give it any point in space and it returns one number, the distance from that
point to the surface of the mug, **negative if the point is inside**. Today we do it in 2D
with a circle and a box, and draw the numbers as pictures.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: the circle and the box
A point `p = (x, y)`. For a circle with centre `c` and radius `r`, the distance from `p` to the
edge is `|p - c| - r`: positive outside, zero on the edge, negative inside (the most negative
value, `-r`, is at the centre).

For a box with centre `c` and half-sizes `(a, b)`, let `q = |p - c| - (a, b)` (per coordinate).
If both parts of `q` are negative the point is inside, and the distance to the nearest side
is `max(q)`. Otherwise it is outside, and the distance is the length of the positive parts of `q`.

In [ ]:
def sd_circle(p, c, r):
    """Signed distance from points p (shape (..., 2)) to a circle of centre c and radius r."""
    p = np.asarray(p, dtype=float)
    return np.linalg.norm(p - np.asarray(c, dtype=float), axis=-1) - r


def sd_box(p, c, half):
    """Signed distance from points p (shape (..., 2)) to a box with centre c and half-sizes half."""
    q = np.abs(np.asarray(p, dtype=float) - np.asarray(c, dtype=float)) - np.asarray(half, dtype=float)
    outside = np.linalg.norm(np.maximum(q, 0.0), axis=-1)
    inside = np.minimum(np.max(q, axis=-1), 0.0)
    return outside + inside

## Step 2: join two shapes, and turn numbers into a picture
A point is in the *union* of two shapes when it is inside at least one of them, so we take the
smaller of the two numbers: `min(d1, d2)`. To draw a shape we evaluate the function on a grid
of points and colour each cell by its sign.

In [ ]:
def union(d1, d2):
    return np.minimum(d1, d2)


def make_grid(xlim, ylim, n):
    """n by n cell centres over the rectangle; returns the points with shape (n, n, 2)."""
    x = xlim[0] + (np.arange(n) + 0.5) * (xlim[1] - xlim[0]) / n
    y = ylim[0] + (np.arange(n) + 0.5) * (ylim[1] - ylim[0]) / n
    X, Y = np.meshgrid(x, y)
    return np.stack([X, Y], axis=-1)


def inside_fraction(f, xlim, ylim, n):
    """Fraction of grid cells whose centre is inside the shape (f < 0)."""
    return float(np.mean(f(make_grid(xlim, ylim, n)) < 0))

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    xlim, ylim = (-2.0, 3.5), (-1.5, 1.5)
    P = make_grid(xlim, ylim, 300)
    circle = sd_circle(P, (0, 0), 1.0)
    box = sd_box(P, (2.0, 0.0), (0.5, 0.5))
    both = union(circle, box)

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
    for ax, d, name in zip(axes, [circle, box, both], ["circle: |p - c| - r", "box", "union = min(circle, box)"]):
        im = ax.imshow(d, extent=[*xlim, *ylim], origin="lower", cmap="RdBu", vmin=-1.5, vmax=1.5)
        ax.contour(P[..., 0], P[..., 1], d, levels=[0.0], colors="k", linewidths=2)
        ax.set_title(name)
        ax.set_xlabel("x (units)")
    axes[0].set_ylabel("y (units)")
    fig.colorbar(im, ax=axes, label="signed distance (units): blue negative = inside", shrink=0.85)
    fig.savefig(OUT / "sdf_pictures.png", dpi=110, bbox_inches="tight")

    x = np.linspace(-2.0, 3.5, 400)
    line = np.stack([x, np.full_like(x, 0.3)], axis=-1)
    fig2, ax2 = plt.subplots(figsize=(6.5, 3.8))
    ax2.plot(x, sd_circle(line, (0, 0), 1.0), label="circle")
    ax2.plot(x, sd_box(line, (2.0, 0.0), (0.5, 0.5)), label="box")
    ax2.plot(x, union(sd_circle(line, (0, 0), 1.0), sd_box(line, (2.0, 0.0), (0.5, 0.5))), "k--", label="union (min)")
    ax2.axhline(0, color="gray", lw=1)
    ax2.set_xlabel("x along the line y = 0.3 (units)")
    ax2.set_ylabel("signed distance (units)")
    ax2.set_title("Where a curve is below zero, the point is inside")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "sdf_profile.png", dpi=120)

    frac = inside_fraction(lambda q: sd_circle(q, (0, 0), 1.0), (-1.5, 1.5), (-1.5, 1.5), 300)
    print("circle area by counting:", round(frac * 9.0, 4), " exact pi r^2 =", round(np.pi, 4))
    p = np.array([1.2, 0.3])
    print("worked example: circle", round(float(sd_circle(p, (0, 0), 1.0)), 3),
          " box", round(float(sd_box(p, (2.0, 0.0), (0.5, 0.5))), 3),
          " union", round(float(union(sd_circle(p, (0, 0), 1.0), sd_box(p, (2.0, 0.0), (0.5, 0.5)))), 3))

## Where this stops
We only measured distances and joined two shapes. Session 2 adds cutting (subtraction), smooth
blending and a parametric mug cross-section you can resize.

## Coding tasks
1. **Easy.** Change the circle and box in `lesson.py` (move the box to (3, 0)) and predict the value of the union at the origin before you run it.
2. **Stretch.** Write `sd_segment_band(p, half)` for the strip |y| < half (value |y| - half) and union it with the circle; draw it with the same plotting code.
3. **Explain.** Explain why the value at the exact centre of a circle is -r, and where a box has its most negative value.

## Assessment
- **A. Conceptual.** Why do we use a negative value for points inside a shape, rather than just the distance to the surface?
- **B. Numerical.** A circle of radius 2 is centred at the origin. What is the signed distance of the point (3, 4)? Is it inside?
- **C. Graph and simulation interpretation.** In the profile graph along the line y = 0.3, between which x values is the union inside? Why are there two separate stretches?
- **D. Design and optimisation challenge.** Design: the box has centre (2, 0) and half-sizes (0.5, 0.5). Choose the radius of a circle centred at the origin so that the two shapes just touch, then so that they overlap. Which is better for a real part and why?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    c0 = (0.0, 0.0)
    checks.append(("circle: point (3, 4) is 3 units outside a circle of radius 2", abs(float(L.sd_circle((3, 4), c0, 2.0)) - 3.0) < 1e-12))
    checks.append(("circle: the centre has value -r (negative = inside)", abs(float(L.sd_circle(c0, c0, 2.0)) + 2.0) < 1e-12))
    checks.append(("circle: a point on the edge has value 0", abs(float(L.sd_circle((0, 2), c0, 2.0))) < 1e-12))
    checks.append(("box: (3, 0) is 2 units from a box of half-size 1", abs(float(L.sd_box((3, 0), c0, (1, 1))) - 2.0) < 1e-12))
    checks.append(("box: the centre of a box of half-size 1 has value -1", abs(float(L.sd_box(c0, c0, (1, 1))) + 1.0) < 1e-12))
    checks.append(("box: a corner point (2, 2) is sqrt(2) from the box", abs(float(L.sd_box((2, 2), c0, (1, 1))) - np.sqrt(2.0)) < 1e-12))

    # worked example: circle (0,0) r=1, box centre (2,0) half (0.5,0.5), point (1.2, 0.3)
    p = np.array([1.2, 0.3])
    dc = float(L.sd_circle(p, c0, 1.0))
    db = float(L.sd_box(p, (2.0, 0.0), (0.5, 0.5)))
    du = float(L.union(dc, db))
    checks.append((f"worked example: circle {dc:.3f}, box {db:.3f}, union = min = {du:.3f}",
                   abs(dc - (np.sqrt(1.53) - 1.0)) < 1e-12 and abs(db - 0.3) < 1e-12 and abs(du - 0.237) < 1e-3))

    # the union is inside where either shape is inside
    pts = np.array([[0.0, 0.0], [2.0, 0.0], [1.2, 1.2]])
    inside = L.union(L.sd_circle(pts, c0, 1.0), L.sd_box(pts, (2.0, 0.0), (0.5, 0.5))) < 0
    checks.append(("union: inside the circle, inside the box, outside both", list(inside) == [True, True, False]))

    # the profile along y = 0.3 (Section C) and the touching/overlapping design (Section F)
    xs = np.array([-0.96, -0.95, 0.95, 0.96, 1.49, 1.51, 2.49, 2.51])
    line = np.stack([xs, np.full_like(xs, 0.3)], axis=-1)
    prof = L.union(L.sd_circle(line, c0, 1.0), L.sd_box(line, (2.0, 0.0), (0.5, 0.5))) < 0
    checks.append(("profile along y = 0.3: inside for -0.954 < x < 0.954 and for 1.5 < x < 2.5",
                   list(prof) == [False, True, True, False, False, True, True, False]))
    checks.append(("design: a circle of radius 1.5 at the origin touches the box at (1.5, 0) (both values 0)",
                   abs(float(L.sd_circle((1.5, 0.0), c0, 1.5))) < 1e-12 and abs(float(L.sd_box((1.5, 0.0), (2.0, 0.0), (0.5, 0.5)))) < 1e-12))
    checks.append(("design: radius 1.6 overlaps the box, the point (1.5, 0) has value -0.1 in the circle",
                   abs(float(L.sd_circle((1.5, 0.0), c0, 1.6)) + 0.1) < 1e-12))
    # the app's challenge: a point at the circle centre is inside
    checks.append(("app challenge: the point (-0.6, 0) is inside a circle of radius 0.5 centred at (-0.6, 0)",
                   float(L.sd_circle((-0.6, 0.0), (-0.6, 0.0), 0.5)) < 0))

    # a signed distance changes by at most the step you take
    rng = np.random.default_rng(0)
    a = rng.uniform(-3, 3, (200, 2))
    e = rng.normal(size=(200, 2)) * 0.1
    d1 = L.sd_circle(a, c0, 1.0) - L.sd_circle(a + e, c0, 1.0)
    d2 = L.sd_box(a, c0, (1.0, 0.5)) - L.sd_box(a + e, c0, (1.0, 0.5))
    step = np.linalg.norm(e, axis=1)
    checks.append(("a step of length s changes the distance by at most s", bool(np.all(np.abs(d1) <= step + 1e-12) and np.all(np.abs(d2) <= step + 1e-12))))

    # counting grid cells approximates the area pi r^2
    frac = L.inside_fraction(lambda q: L.sd_circle(q, c0, 1.0), (-1.5, 1.5), (-1.5, 1.5), 300)
    checks.append((f"counting cells: circle area {frac * 9.0:.4f} is within 0.5% of pi", abs(frac * 9.0 - np.pi) / np.pi < 0.005))
    # edge: a grid with all cells outside gives 0
    checks.append(("edge case: a shape far outside the grid has fraction 0",
                   L.inside_fraction(lambda q: L.sd_circle(q, (10, 10), 1.0), (-1, 1), (-1, 1), 20) == 0.0))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
